# 推理加速与 KV Cache

> 前面我们了解了模型如何一步步生成文本。但真上手跑起来，一个 7B 模型的权重用 BF16 存大约 14 GB，一张数据中心的 GPU 每秒能做几百万亿次浮点运算——两个数字放一起，你肯定以为生成快得看不清，实际用起来每秒往往只输出几十个 Token。
>
> 为什么算力这么强，速度却这么慢？慢的原因不在算力，在「数据怎么搬」。这就是本章要拆开看的问题。
>
> 本章介绍推理加速的四组核心内容：
>
> 1. **Prefill 与 Decode**：一次请求分两段，负载完全不同。
> 2. **KV Cache**：历史 K/V 不必重算，缓存让 Decode 只处理新 Token。
> 3. **KV Cache 的代价**：缓存自己吃显存；GQA / MQA 来救场。
> 4. **memory-bound**：Decode 慢是因为“喂不饱”，不是“算不快”。
>
> 读完这一章，你会明白一次推理里 Prompt 进来后模型做了什么、生成为何串行、KV Cache 怎么省重复计算又怎么变成负担，以及那些优化名词到底针对哪个环节。

我们先来看一次完整的请求是什么样。假设你发过去一段 2000 个 Token 的 Prompt，模型会先把它整体读一遍，再一个 Token 接一个 Token 地往外生成。

这两个阶段的计算模式完全不同。换句话说，后面所有的推理优化，都建立在区分它们的基础上。

## 1. Prefill 与 Decode

第一段叫 **Prefill**。为什么这么叫？因为 2000 个 Prompt Token 一起进模型，一次算完所有位置的 Attention 和 FFN。这是一次大矩阵运算，几千个 Token 互相并行，正好是 GPU 最喜欢的形状，算力可以吃满。Prefill 结束，模型交出第一个输出 Token。

第二段叫 **Decode**。从第二个 Token 开始，每一步只有一个新 Token 进模型。矩阵从「2000 × hidden」缩成「1 × hidden」，非常瘦，算力再也吃不满。但是每一步都要把整套权重和越来越长的历史读一遍。生成 300 个 Token，就是 300 次这样的串行小步。

```text
Prompt tokens ── Prefill ──> 第一个 Token
                           ↓
                      Decode step
                           ↓
                      Decode step
                           ↓
                          ...
```

从用户体感看，这两段对应两种不同的「慢」。

```text
发出请求 ───────── 第一个字出现 ── 逐字逐字往外蹦 ──── 结束
        ←── TTFT ──→ ←─ 每个 TPOT 出一个 ─→
```

**TTFT**（Time To First Token，首 Token 延迟）是从发出请求到第一个 Token 出现的等待，由排队和 Prefill 主导。**TPOT**（Time Per Output Token）是之后 Token 之间的平均间隔，由 Decode 主导。如果用户说“等了半天才冒出第一个字”，那是 TTFT 的问题；要是“吐字后一个一个蹦得慢”，那就是 TPOT 的问题。换句话说，两个指标混成一个「延迟」，就没法定位瓶颈了。

In [ ]:
prompt_tokens = 2000
output_tokens = 300

print("Prefill: 一次性处理", prompt_tokens, "个 prompt token（大矩阵，算力吃满）")
print("Decode : 之后执行", output_tokens, "次串行 step（每次 1 个 token，读全部权重）")
print()
print("关键观察：长 prompt 主要推高 TTFT；长输出主要推高 TPOT 和总时延")

## 2. 重复计算的问题

Decode 每一步只进来 1 个新 Token，但 Attention 要求它和**全部历史**做运算。新 Token 的 Q/K/V 需要当前层、当前词的隐藏状态。那么问题来了：要拿到这个隐藏状态，得把整个前缀重新过一遍模型。

```text
Step 1: [A]              -> token1
Step 2: [A, token1]      -> token2   <- A 被重新算了第 2 遍
Step 3: [A, token1, t2]  -> token3   <- A 第 3 遍，token1 第 2 遍
```

生成到第 N 个 Token 时，历史前缀累计被处理了 $1 + 2 + \dots + N \approx N^2/2$ 次。换句话说，下面这个实验用「重复处理的历史 token 数」作代理量，让你感受一下这个平方增长。

In [ ]:
def repeated_prefix_proxy(n):
    """不缓存时，生成 n 个 token 累计处理的历史 token 数（1+2+...+n）"""
    return n * (n + 1) // 2

for n in [10, 100, 1000]:
    naive = repeated_prefix_proxy(n)
    cached = n
    print(f"N={n:4d}  不缓存累计处理 {naive:8d} 个历史 token | 缓存后 {cached:5d} 个")

print()
print("关键观察：不缓存时是平方增长；长度翻 10 倍，浪费翻约 100 倍")

这里有一个关键的观察：**历史 Token 的 K/V 根本不会变。** 举个例子，第 100 步算出的「巴黎」的 K/V，到第 101 步、第 200 步还是同一组数。为什么？因为同一个序列里，历史只增不改。

既然算过一遍就不变，为什么要一遍遍重算？答案显然是：不该重算。把它存起来，就是下一节的主角。

## 3. KV Cache 的原理

**KV Cache 的定义：把 Attention 已经算出的历史 Key / Value 向量存进显存，后续步骤直接取用，不再重算。** 说通俗点，就是给每个 Token 的「记忆」留一份底稿，新 Token 来了直接查表。

有了 KV Cache，Decode 每一步真正要算的只剩两件小事。第一件，新 Token 自己的 Q、K、V（各 1 份）。第二件，拿新的 Q 和缓存里全部 K/V 做一次 Attention。矩阵从「整段前缀 × hidden」缩成「1 × hidden」。

但是，有一句话必须说在前面，这也是最常见的误解：**KV Cache 不是把 Attention 从 $O(N^2)$ 变成 $O(N)$ 的魔法。** 为什么？因为新 Query 仍要和越来越长的历史 KV 做注意力，这部分计算一步没少。缓存省掉的只是「历史 K/V 的重复计算」。换句话说，它用显存换计算，这笔交易划不划算，取决于显存账有多大，下一节就来算。

顺手打一个标记：后面你会遇到 FlashAttention 和 PagedAttention 两个名字。FlashAttention 优化的是 Attention 计算本身的显存读写（Kernel 层）；PagedAttention 管的是 KV Cache 这块缓存怎么分配（系统层，推理系统一本展开）。换句话说，它们和「KV Cache 要不要存在」是三件不同的事。

## 4. KV Cache 的显存开销

缓存不是免费的，它住在显存里。我们先看一个粗略的估算公式：

$$
\text{KV bytes}
\approx
2 \times L \times T \times H_{kv} \times D \times B \times \text{bytes}
$$

这里的 `2` 表示 K 和 V 各一份，`L` 是层数，`T` 是上下文长度，`H_kv` 是 KV head 数，`D` 是每个 head 的维度，`B` 是并发请求数。

先别运行代码，拿笔算一个真实配置。某个 7B 级模型，32 层，GQA 8 个 KV head，head_dim 128，batch 1，上下文 8192，BF16（每个数 2 字节）：

```text
2 (K和V) × 32 (层) × 8192 (token) × 8 (KV head) × 128 (head_dim) × 2 (字节) × 1 (batch)
= 1,073,741,824 字节 ≈ 1.07 GB
```

算出来什么结果？一个请求、8K 上下文，就要 1 GB 出头。那如果把上下文换成 128K（放大 16 倍）呢？约 17.2 GB，比模型权重还大。换句话说，这就是长上下文和多用户并发最先撞上的墙。下面用代码验证这笔账，并把 batch 拉到 16，看看多请求时的情形。

In [ ]:
def kv_cache_gb(layers, tokens, kv_heads, head_dim, batch, bytes_per_elem=2):
    """按公式估算 KV Cache 大小（GB）"""
    total = 2 * layers * tokens * kv_heads * head_dim * batch * bytes_per_elem
    return total / 1e9

# 验证手算：7B 级模型、GQA 8 KV head、8K/128K 上下文
print(f"8K 上下文, batch 1  : {kv_cache_gb(32, 8192, 8, 128, 1):.2f} GB  (手算 1.07)")
print(f"128K 上下文, batch 1: {kv_cache_gb(32, 131072, 8, 128, 1):.2f} GB  (手算 17.2)")
print()

# 同一模型、batch 16，对比三种 KV head 配置
cfgs = [("MHA 32 KV heads", 32), ("GQA 8 KV heads", 8), ("MQA 1 KV head", 1)]
for name, kvh in cfgs:
    size = kv_cache_gb(32, 8192, kvh, 128, batch=16)
    print(f"{name:<18}: {size:6.2f} GB")

print()
print("关键观察：MHA 和 MQA 相差 32 倍——差距只来自 KV head 数")

表格里最后一列的对比藏着一个大问题。同一张 24 GB 的卡，跑 GQA 版 8K 上下文还能容纳 16 个并发请求，跑 MHA 版连 4 个都危险。为什么？因为模型本身一个参数都没变，差的只是「K/V 存了几份」。

下一节就来拆这个 32 倍差距是怎么来的。

## 5. MHA、GQA 与 MQA

标准的多头注意力（MHA）里，32 个 Query head 各配一套自己的 K/V head，一共 32 套。回头看上一节的公式，缓存大小和 KV head 数成正比。那么问题来了——

> **每个 Query head，真的都需要一份独占的 K/V 吗？**

实验给出的答案有点意外：不需要。把若干 Query head 分组，让同组共享一套 K/V，模型质量几乎不掉，KV Cache 却成倍缩小。这就是 **GQA**（Grouped-Query Attention）：8 个 KV head 服务 32 个 Query head，每 4 个 Query 共享一套 K/V。再极端一点，所有 Query head 共享唯一一套 K/V，就是 **MQA**（Multi-Query Attention）。

```text
MHA: 32 个 Query head，32 套 K/V   <- 每人独占
GQA: 32 个 Query head， 8 套 K/V   <- 每 4 人共享一套
MQA: 32 个 Query head， 1 套 K/V   <- 全员共享
```

回头看上一节的对比：MHA → GQA 缩小 4 倍，MHA → MQA 缩小 32 倍。为什么能省这么多？不是靠压缩数值，而是靠「共享」。现代开源模型（LLaMA 3、Qwen 系列）几乎都用 GQA；DeepSeek 则更激进，用 MLA 把 K/V 压缩成低秩向量，那是 Part 2「KV Cache 及架构演进」一本讲过的话题。换句话说，这也是为什么招聘 JD 里这三个词经常并排出现：它们回答的是同一个问题——K/V 存几份。

## 6. Decode 的带宽瓶颈

现在回答本章开头的问题：GPU 每秒几百万亿次运算，为什么每秒只生成几十个 Token？

我们先算 Decode 一步的计算量。1 个 Token 穿过整个模型，粗略估算 FLOPs ≈ 2 × 参数量 ≈ 1.4 × 10¹⁰，对任何现代 GPU 都是微秒级的活。但是计算之前，数据得先到位：14 GB 权重加上 KV Cache，每一步都要从显存搬进计算单元。

显存带宽是有限的，按 2 TB/s 算，光搬 14 GB 就要 7 毫秒。这意味着什么？这一步的速度上限被带宽钉死在每秒约 140 个 Token，而算力利用率不到 1%。

一言以蔽之：**Decode 是 memory-bound——瓶颈是「喂不饱」，不是「算不快」。** Prefill 恰好相反，几千个 Token 的大矩阵让算力成为瓶颈（compute-bound）。这个对比是后面所有优化的出发点，三条路线各自针对一个症状：

```text
权重太大、每步搬得太慢   -> 量化（下一章）
一次只能确认一个 Token    -> 投机解码
多请求挤一张卡           -> 推理系统
```

In [ ]:
params = 7e9

weight_bf16 = params * 2          # BF16 每参数 2 字节
weight_int4 = params * 0.5        # INT4 每参数 0.5 字节
gpu_bw = 2e12                     # 显存带宽 2 TB/s（量级示意）

print(f"7B BF16 权重: {weight_bf16/1e9:.0f} GB -> 每步搬运 {weight_bf16/gpu_bw*1000:.1f} ms")
print(f"7B INT4 权重: {weight_int4/1e9:.1f} GB -> 每步搬运 {weight_int4/gpu_bw*1000:.1f} ms")
print()
print("关键观察：权重小 4 倍，每步搬运时间也差不多小 4 倍——")
print("量化提速 Decode 的原理，就是让每一步搬的数据变少")

## 小结

这一章把「推理慢」拆成了能定位的具体环节。一次请求分两段：Prefill 大矩阵、算力吃满；Decode 小步串行、反复读权重。

谈优化先对准指标，因为 TTFT 对应 Prefill，TPOT 对应 Decode。KV Cache 把不变的历史 K/V 留在显存，省掉平方级的重复计算，但它不是把 Attention 变成 O(N)——新 Query 仍要扫全部历史。

另外，KV Cache 自己会长成显存大户；GQA / MQA 靠「共享 K/V」把它缩小 4 到 32 倍。换句话说，Decode 是 memory-bound：瓶颈在数据搬运，不在算力——这是量化的动机。

下一章就沿着最后这条线走：

> **模型权重太大、每步搬运太贵，怎么办？**

## 作业

后面的三道题，本质都是让你算一笔账：KV Cache 有多大、GQA 省多少、Decode 为什么吃带宽。

> **关于 AI 辅助**：你可以让 AI 给点思路、把步骤拆开，但最好别直接交给他做完。
> 这些账目在排障和面试里都属于心算级工具，自己动手算一遍才划算。

### 作业 1：估算 KV Cache 的大小

我们要把公式变成函数。字节数 ≈ `2 × 层数 × token 数 × KV head 数 × head_dim × batch × 每元素字节数`。

**小提示**：函数返回 GB 时记得除以 `1e9`。

In [ ]:
# 作业 1：KV Cache 手算验证 填空

def kv_cache_gb(layers, tokens, kv_heads, head_dim, batch, bytes_per_elem=2):
    """返回 KV Cache 的 GB 数，2 表示 K 和 V 各占一份"""
    # TODO：把下面三引号里的内容替换成你的代码
    """按 2 × layers × tokens × kv_heads × head_dim × batch × bytes_per_elem 算出 GB 数"""

# 7B 级模型、GQA 8 个 KV head、128K 上下文、batch 1、BF16
assert abs(kv_cache_gb(32, 131072, 8, 128, 1) - 17.18) < 0.01
print("✅ 作业 1 通过：你现在能估算任意配置的 KV Cache 大小")

### 作业 2：GQA / MQA 到底省多少

KV Cache 的大小和 KV head 数成正比，这正是 GQA、MQA 的全部显存收益来源。

**小提示**：两种配置的 KV head 数相除，就是 KV Cache 的缩小倍数。

In [ ]:
# 作业 2：KV head 数与显存 填空

def kv_ratio(kv_heads_a, kv_heads_b):
    """返回用 kv_heads_a 与 kv_heads_b 两种配置时 KV Cache 大小的比值"""
    # TODO：把下面三引号里的内容替换成你的代码
    """KV Cache 大小与 KV head 数成正比，返回 a / b"""

assert kv_ratio(32, 8) == 4.0    # MHA -> GQA(8)
assert kv_ratio(32, 1) == 32.0   # MHA -> MQA
print("✅ 作业 2 通过：MHA -> GQA -> MQA 的显存收益你已经会算了")

### 作业 3：算一算 Decode 的 arithmetic intensity

想象一个 $4096 \times 4096$ 的权重矩阵做矩阵乘。FLOPs 约 `2 × batch × 4096 × 4096`；但权重只需要从显存搬一次，字节约 `4096 × 4096 × 2`。FLOPs 与 bytes 的比值叫 arithmetic intensity，它决定 GPU 是「算不过来」还是「喂不饱」。

**小提示**：batch 越大，同样的权重字节换来的计算越多，比值就越高。

In [ ]:
# 作业 3：arithmetic intensity 填空

def arithmetic_intensity(batch, bytes_per_elem=2):
    """4096×4096 矩阵乘一次的 FLOPs / 权重搬运字节数"""
    flops = 2 * batch * 4096 * 4096
    weight_bytes = 4096 * 4096 * bytes_per_elem
    # TODO：把下面三引号里的内容替换成你的代码
    """返回 flops 除以 weight_bytes 的结果"""

assert arithmetic_intensity(1) == 1.0
assert arithmetic_intensity(64) == 64.0
print("✅ 作业 3 通过：batch 越大每字节权重换来的计算越多，这就是 Decode 靠并发吃饱带宽的直觉")